In [127]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
import re 

# Monthly Trading

In [128]:
df = pd.read_csv(
    "/Users/Thomas/Desktop/Master Thesis/Data/Stock_Monthly_Trading.csv",
    sep=";"
)

In [129]:
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

Dataset Shape: (1894, 387)

Column Names:
['Company name Latin alphabet', 'Risk level', 'BvD ID number', 'Monthly - Trading volume - January\nLast avail. yr', 'Monthly - Trading volume - January\nYear - 1', 'Monthly - Trading volume - January\nYear - 2', 'Monthly - Trading volume - January\nYear - 3', 'Monthly - Trading volume - January\nYear - 4', 'Monthly - Trading volume - January\nYear - 5', 'Monthly - Trading volume - January\nYear - 6', 'Monthly - Trading volume - January\nYear - 7', 'Monthly - Trading volume - January\nYear - 8', 'Monthly - Trading volume - January\nYear - 9', 'Monthly - Trading volume - January\nYear - 10', 'Monthly - Trading volume - January\nYear - 11', 'Monthly - Trading volume - January\nYear - 12', 'Monthly - Trading volume - January\nYear - 13', 'Monthly - Trading volume - January\nYear - 14', 'Monthly - Trading volume - January\nYear - 15', 'Monthly - Trading volume - January\n2026', 'Monthly - Trading volume - January\n2025', 'Monthly - Trading volume -

In [130]:
absolute_year_pattern = re.compile(r'Monthly - Trading volume - (\w+)\n(20\d{2})')
melt_cols = [c for c in df.columns if absolute_year_pattern.search(c)]

In [131]:
df_long = pd.melt(
    df, 
    id_vars=['Company name Latin alphabet', 'BvD ID number'], 
    value_vars=melt_cols, 
    var_name='Raw', 
    value_name='Volume'
)

In [132]:
df_long[['Month', 'Year']] = df_long['Raw'].str.extract(r'Monthly - Trading volume - (\w+)\n(20\d{2})')

# 5. Clean Numeric Values (remove dots and convert)
df_long['Volume'] = df_long['Volume'].str.replace('.', '', regex=False)
df_long['Volume'] = pd.to_numeric(df_long['Volume'], errors='coerce')

# 6. Sort for time-series consistency
df_long = df_long.sort_values(['BvD ID number', 'Year', 'Month'])

In [133]:
df_long.head()

,Company name Latin alphabet,BvD ID number,Raw,Volume,Month,Year
120049,Rever Events L.L.C,AE0001327927,Monthly - Trading volume - April\n2011,NaN,April,2011
241265,Rever Events L.L.C,AE0001327927,Monthly - Trading volume - August\n2011,NaN,August,2011
362481,Rever Events L.L.C,AE0001327927,Monthly - Trading volume - December\n2011,NaN,December,2011
59441,Rever Events L.L.C,AE0001327927,Monthly - Trading volume - February\n2011,NaN,February,2011
29137,Rever Events L.L.C,AE0001327927,Monthly - Trading volume - January\n2011,NaN,January,2011


In [134]:
len(df_long)

363648

## Calculating volatility, shock and trend 

In [135]:
# Ensure Year and Value are numeric
df_long['Year'] = pd.to_numeric(df_long['Year'])
# If Value still has 'n.a.' or dots, this cleans it one last time
df_long['Volume'] = pd.to_numeric(df_long['Volume'].astype(str).str.replace('.', '', regex=False), errors='coerce')

In [136]:
# Group by Supplier and Year
df_signals = df_long.groupby(['BvD ID number', 'Company name Latin alphabet','Year'])['Volume'].agg([
    ('avg_vol', 'mean'),
    ('std_vol', 'std'),
    ('max_vol', 'max'),
    ('min_vol', 'min')
]).reset_index()

# Create 'Stability' (Volatility) and 'Shock' signals
# vol_stability: High = unstable/risky behavior
df_signals['vol_stability_score'] = df_signals['std_vol'] / df_signals['avg_vol']

# vol_shock: High = a single month had a massive surge (Risk of supply chain panic)
df_signals['vol_shock_ratio'] = df_signals['max_vol'] / (df_signals['min_vol'] + 1)

In [137]:
# For every group, we calculate the slope of the 12 months
# This tells the model: Is market interest in this supplier dying or growing?
trends = []

for name, group in df_long.groupby(['BvD ID number', 'Year']):
    valid = group.dropna(subset=['Volume']).sort_values('Month') # Sort months chronologically
    if len(valid) >= 2:
        y = valid['Volume'].values
        x = np.arange(len(y))
        slope = np.polyfit(x, y, 1)[0]
        rel_trend = slope / valid['Volume'].mean()
    else:
        rel_trend = np.nan
    
    trends.append({'BvD ID number': name[0], 'Year': name[1], 'vol_trend_slope': rel_trend})

df_trends = pd.DataFrame(trends)

/var/folders/h9/b0m54z_x70z3xs93z4k09jwc0000gn/T/ipykernel_31416/419294518.py:11: RuntimeWarning: invalid value encountered in scalar divide
  rel_trend = slope / valid['Volume'].mean()


In [138]:
# Merge the Volatility/Shock with the Trend
df_final = df_signals.merge(df_trends, on=['BvD ID number', 'Year'])

# CREATE THE JOIN KEY (Leakage Prevention)
# We want 2022 stock data to explain 2023 contract pressure.
df_final['Join_Year'] = df_final['Year'].astype(int) + 1

In [139]:
df_final.tail(20)

,BvD ID number,Company name Latin alphabet,Year,avg_vol,std_vol,max_vol,min_vol,vol_stability_score,vol_shock_ratio,vol_trend_slope,Join_Year
30268,ZA201611084407,RED BOX Marketing (PTY) LTD,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024
30269,ZA201611084407,RED BOX Marketing (PTY) LTD,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
30270,ZA201611084407,RED BOX Marketing (PTY) LTD,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026
30271,ZA201611084407,RED BOX Marketing (PTY) LTD,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2027
30272,ZA202147957107,Aspen SA Operations (PTY) LTD,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012
30273,ZA202147957107,Aspen SA Operations (PTY) LTD,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013
30274,ZA202147957107,Aspen SA Operations (PTY) LTD,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014
30275,ZA202147957107,Aspen SA Operations (PTY) LTD,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015
30276,ZA202147957107,Aspen SA Operations (PTY) LTD,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016
30277,ZA202147957107,Aspen SA Operations (PTY) LTD,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017


In [140]:
len(df_final)

30288

# Monthly Market Cap

In [144]:
df_market_cap = pd.read_csv("/Users/Thomas/Desktop/Master Thesis/Data/Stock_Monthly_Market_Cap.csv",
sep=";"
)

In [145]:
df_market_cap.head()

,Company name Latin alphabet,Risk level,BvD ID number,Monthly - Market Capitalisation - January\nth DKK Last avail. yr,Monthly - Market Capitalisation - January\nth DKK Year - 1,Monthly - Market Capitalisation - January\nth DKK Year - 2,Monthly - Market Capitalisation - January\nth DKK Year - 3,Monthly - Market Capitalisation - January\nth DKK Year - 4,Monthly - Market Capitalisation - January\nth DKK Year - 5,Monthly - Market Capitalisation - January\nth DKK Year - 6,...,Monthly - Market Capitalisation - December\nth DKK 2020,Monthly - Market Capitalisation - December\nth DKK 2019,Monthly - Market Capitalisation - December\nth DKK 2018,Monthly - Market Capitalisation - December\nth DKK 2017,Monthly - Market Capitalisation - December\nth DKK 2016,Monthly - Market Capitalisation - December\nth DKK 2015,Monthly - Market Capitalisation - December\nth DKK 2014,Monthly - Market Capitalisation - December\nth DKK 2013,Monthly - Market Capitalisation - December\nth DKK 2012,Monthly - Market Capitalisation - December\nth DKK 2011
0,Jeev Lifeworks LLP,Do not source,IN0038742908,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
1,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,MY422033-W,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
2,Daicel Corporation,Go ahead,JP4120001125937,15.822.500,15.510.954,19.010.998,13.764.740,13.407.109,13.696.649,20.354.184,...,13.334.310,21.303.160,23.281.440,24.667.305,27.258.712,37.522.834,26.257.920,16.057.724,13.529.696,12.653.192
3,"Polyplastics Co.,Ltd.",Take caution,JP1010401053834,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
4,"Kolon Industries, Inc.",Go ahead,KR1353110013606,6.566.939,3.417.462,5.455.712,6.556.883,9.265.650,6.101.344,6.346.316,...,6.166.066,7.739.139,9.013.737,13.160.487,10.869.171,9.189.113,6.758.854,7.037.893,8.436.620,7.904.583


In [146]:
mcap_pattern = re.compile(r'Market Capitalisation - (\w+)\n.*(20\d{2})')

mcap_cols = [c for c in df_market_cap.columns if mcap_pattern.search(c)]

In [147]:
df_martket_cap_long = pd.melt(
    df_market_cap, 
    id_vars=['BvD ID number', 'Company name Latin alphabet','Risk level'], 
    value_vars=mcap_cols, 
    var_name='Raw', 
    value_name='Value'
)

In [148]:
df_martket_cap_long.head()

,BvD ID number,Company name Latin alphabet,Risk level,Raw,Value
0,IN0038742908,Jeev Lifeworks LLP,Do not source,Monthly - Market Capitalisation - January\nth ...,n.a.
1,MY422033-W,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,Monthly - Market Capitalisation - January\nth ...,n.a.
2,JP4120001125937,Daicel Corporation,Go ahead,Monthly - Market Capitalisation - January\nth ...,15.822.500
3,JP1010401053834,"Polyplastics Co.,Ltd.",Take caution,Monthly - Market Capitalisation - January\nth ...,n.a.
4,KR1353110013606,"Kolon Industries, Inc.",Go ahead,Monthly - Market Capitalisation - January\nth ...,6.566.939


In [149]:
df_martket_cap_long[['Month', 'Year']] = df_martket_cap_long['Raw'].str.extract(r'Market Capitalisation - (\w+)\n.*(20\d{2})')
df_martket_cap_long['Value'] = pd.to_numeric(df_martket_cap_long['Value'].astype(str).str.replace('.', '', regex=False), errors='coerce')

In [150]:
df_martket_cap_long.tail()

,BvD ID number,Company name Latin alphabet,Risk level,Raw,Value,Month,Year
363643,BR62398938000169,Iqvia Solutions DO Brasil Ltda.,Take caution,Monthly - Market Capitalisation - December\nth...,NaN,December,2011
363644,US562445503,"VWR Funding, Inc.",Take caution,Monthly - Market Capitalisation - December\nth...,NaN,December,2011
363645,US127738269L,"ZS Associates, Inc.",Take caution,Monthly - Market Capitalisation - December\nth...,NaN,December,2011
363646,US127510898L,Iqvia Inc.,Take caution,Monthly - Market Capitalisation - December\nth...,NaN,December,2011
363647,NaN,NaN,NaN,Monthly - Market Capitalisation - December\nth...,NaN,December,2011


In [ ]:
df_mcap_signals = df_martket_cap_long.groupby(['BvD ID number', 'Year','Risk level'])['Value'].agg([
    ('avg_market_cap', 'mean'),
    ('market_cap_volatility', lambda x: x.std() / x.mean() if x.mean() != 0 else 0)
]).reset_index()


df_mcap_signals['Year'] = df_mcap_signals['Year'].astype(int) # Force Integer 



# 6. Create Join Year for Meta-Matrix
df_mcap_signals['Join_Year'] = df_mcap_signals['Year'] + 1

In [161]:
df_mcap_signals.head()

,BvD ID number,Year,Risk level,avg_market_cap,market_cap_volatility,Join_Year
0,AE0001327927,2011,Take caution,NaN,NaN,2012
1,AE0001327927,2012,Take caution,NaN,NaN,2013
2,AE0001327927,2013,Take caution,NaN,NaN,2014
3,AE0001327927,2014,Take caution,NaN,NaN,2015
4,AE0001327927,2015,Take caution,NaN,NaN,2016


In [ ]:
# Join on both BvD ID and Year 
df_stocks_joined = df_final.merge(
    df_mcap_signals,
    on=['BvD ID number', 'Year', 'Join_Year'],
    how='left'
)

In [165]:
df_stocks_joined.tail(20)

,BvD ID number,Company name Latin alphabet,Year,avg_vol,std_vol,max_vol,min_vol,vol_stability_score,vol_shock_ratio,vol_trend_slope,Join_Year,Risk level,avg_market_cap,market_cap_volatility
30268,ZA201611084407,RED BOX Marketing (PTY) LTD,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,Take caution,NaN,NaN
30269,ZA201611084407,RED BOX Marketing (PTY) LTD,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025,Take caution,NaN,NaN
30270,ZA201611084407,RED BOX Marketing (PTY) LTD,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,Take caution,NaN,NaN
30271,ZA201611084407,RED BOX Marketing (PTY) LTD,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2027,Take caution,NaN,NaN
30272,ZA202147957107,Aspen SA Operations (PTY) LTD,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012,Take caution,NaN,NaN
30273,ZA202147957107,Aspen SA Operations (PTY) LTD,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013,Take caution,NaN,NaN
30274,ZA202147957107,Aspen SA Operations (PTY) LTD,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014,Take caution,NaN,NaN
30275,ZA202147957107,Aspen SA Operations (PTY) LTD,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015,Take caution,NaN,NaN
30276,ZA202147957107,Aspen SA Operations (PTY) LTD,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016,Take caution,NaN,NaN
30277,ZA202147957107,Aspen SA Operations (PTY) LTD,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017,Take caution,NaN,NaN


# Stocks beta

In [168]:
df_stocks_beta = pd.read_csv('/Users/Thomas/Desktop/Master Thesis/Data/Stocks_beta_correl_etc.csv', 
sep =';'
)

In [169]:
df_stocks_beta.head()

,Company name Latin alphabet,Risk level,Date - Current,Price trends - Last week\n%,Price trends - 4 weeks\n%,Price trends - 13 weeks\n%,Price trends - Quarter to date\n%,Price trends - Previous quarter to date\n%,Price trends - Year to date\n%,Price trends - 52 weeks\n%,...,Market price - Year to date - Low\nDKK,Market price - 52 week - High\nDKK,Market price - 52 week - Low\nDKK,Shares outstanding,Current market capitalisation\nth DKK,Date of last 12 months - EPS,Earnings per share\nDKK,Book value per share\nDKK,BvD ID number,BvD sectors
0,Jeev Lifeworks LLP,Do not source,NaN,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,NaN,n.a.,n.a.,IN0038742908,Business Services
1,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,NaN,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,NaN,n.a.,n.a.,MY422033-W,"Chemicals, Petroleum, Rubber & Plastic"
2,Daicel Corporation,Go ahead,04/03/2026,-11,-7,12,4,4,3,8,...,57,68,43,266.942.682,15.822.500,31/03/2025,7,55,JP4120001125937,"Chemicals, Petroleum, Rubber & Plastic"
3,"Polyplastics Co.,Ltd.",Take caution,NaN,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,NaN,n.a.,n.a.,JP1010401053834,"Chemicals, Petroleum, Rubber & Plastic"
4,"Kolon Industries, Inc.",Go ahead,04/03/2026,-18,1,24,24,32,31,62,...,177,310,114,27.519.091,6.650.519,31/12/2024,16,575,KR1353110013606,"Chemicals, Petroleum, Rubber & Plastic"


In [181]:
beta_columns_to_keep = [
    'BvD ID number',
    'BvD sectors',
    'Risk level',
    'Price trends - 52 weeks\n%',
    'Ref. index 1Beta1 year',
    'Earnings per share\nDKK',
    'Book value per share\nDKK',
    'Shares outstanding',
    'Current market capitalisation\nth DKK'
]

In [182]:
df_stocks_beta_cleaned = df_stocks_beta[beta_columns_to_keep].copy()

In [183]:
cols_to_fix = [
    'Price trends - 52 weeks\n%', 
    'Ref. index 1Beta1 year', 
    'Earnings per share\nDKK', 
    'Book value per share\nDKK'
]

In [185]:
for col in cols_to_fix:
    # Convert to string, remove dots, and convert to numeric (n.a. becomes NaN)
    df_stocks_beta_cleaned [col] = df_stocks_beta_cleaned [col].astype(str).str.replace('.', '', regex=False)
    df_stocks_beta_cleaned [col] = pd.to_numeric(df_stocks_beta_cleaned [col], errors='coerce')

In [186]:
df_stocks_beta_cleaned.head()

,BvD ID number,BvD sectors,Risk level,Price trends - 52 weeks\n%,Ref. index 1Beta1 year,Earnings per share\nDKK,Book value per share\nDKK,Shares outstanding,Current market capitalisation\nth DKK
0,IN0038742908,Business Services,Do not source,NaN,NaN,NaN,NaN,n.a.,n.a.
1,MY422033-W,"Chemicals, Petroleum, Rubber & Plastic",Go ahead,NaN,NaN,NaN,NaN,n.a.,n.a.
2,JP4120001125937,"Chemicals, Petroleum, Rubber & Plastic",Go ahead,8.0,1.0,7.0,55.0,266.942.682,15.822.500
3,JP1010401053834,"Chemicals, Petroleum, Rubber & Plastic",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.
4,KR1353110013606,"Chemicals, Petroleum, Rubber & Plastic",Go ahead,62.0,1.0,16.0,575.0,27.519.091,6.650.519


In [196]:
# Just renaming
df_stocks_beta_cleaned = df_stocks_beta_cleaned.rename(columns={
    'BvD sectors': 'supplier_sector',
    'Risk level': 'moodys_risk_rating',
    'Ref. index 1Beta1 year': 'market_beta_1y',
    'Current market capitalisation\nth DKK': 'Current_market_capitalisation_DKK',
    'Book value per share\nDKK': 'Book_value_per_share_DKK',
    'Price trends - 52 weeks\n%': 'Price_trends_52 weeks_%',
    'Earnings per share\nDKK':'Earnings_per_share_DKK',
    'Price trends - 52 weeks\n%': 'Price_trends_52_weeks_%'
})


print(df_stocks_beta_cleaned.info())

<class 'pandas.DataFrame'>
RangeIndex: 1894 entries, 0 to 1893
Data columns (total 9 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   BvD ID number                      1893 non-null   str    
 1   supplier_sector                    1779 non-null   str    
 2   moodys_risk_rating                 1893 non-null   str    
 3   Price_trends_52 weeks_%            87 non-null     float64
 4   market_beta_1y                     87 non-null     float64
 5   Earnings_per_share_DKK             87 non-null     float64
 6   Book_value_per_share_DKK           86 non-null     float64
 7   Shares outstanding                 1893 non-null   str    
 8   Current_market_capitalisation_DKK  1893 non-null   str    
dtypes: float64(4), str(5)
memory usage: 133.3 KB
None


In [197]:
df_stocks_beta_cleaned.head(10)

,BvD ID number,supplier_sector,moodys_risk_rating,Price_trends_52 weeks_%,market_beta_1y,Earnings_per_share_DKK,Book_value_per_share_DKK,Shares outstanding,Current_market_capitalisation_DKK
0,IN0038742908,Business Services,Do not source,NaN,NaN,NaN,NaN,n.a.,n.a.
1,MY422033-W,"Chemicals, Petroleum, Rubber & Plastic",Go ahead,NaN,NaN,NaN,NaN,n.a.,n.a.
2,JP4120001125937,"Chemicals, Petroleum, Rubber & Plastic",Go ahead,8.0,1.0,7.0,55.0,266.942.682,15.822.500
3,JP1010401053834,"Chemicals, Petroleum, Rubber & Plastic",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.
4,KR1353110013606,"Chemicals, Petroleum, Rubber & Plastic",Go ahead,62.0,1.0,16.0,575.0,27.519.091,6.650.519
5,KR1713110003504,"Chemicals, Petroleum, Rubber & Plastic",Go ahead,75.0,1.0,5.0,37.0,38.000.000,1.762.494
6,GB10348805,Business Services,Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.
7,US358439945L,Business Services,Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.
8,SE5568922727,Wholesale,Go ahead,NaN,NaN,NaN,NaN,n.a.,n.a.
9,EE11107818,Business Services,Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.


# Join stocks and beta

In [198]:
df_financial_view = df_stocks_joined.merge(
    df_stocks_beta_cleaned,
    on=['BvD ID number'],
    how='left'
)



In [199]:
df_financial_view.head()

,BvD ID number,Company name Latin alphabet,Year,avg_vol,std_vol,max_vol,min_vol,vol_stability_score,vol_shock_ratio,vol_trend_slope,...,avg_market_cap,market_cap_volatility,supplier_sector,moodys_risk_rating,Price_trends_52 weeks_%,market_beta_1y,Earnings_per_share_DKK,Book_value_per_share_DKK,Shares outstanding,Current_market_capitalisation_DKK
0,AE0001327927,Rever Events L.L.C,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"Travel, Personal & Leisure",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.
1,AE0001327927,Rever Events L.L.C,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"Travel, Personal & Leisure",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.
2,AE0001327927,Rever Events L.L.C,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"Travel, Personal & Leisure",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.
3,AE0001327927,Rever Events L.L.C,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"Travel, Personal & Leisure",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.
4,AE0001327927,Rever Events L.L.C,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"Travel, Personal & Leisure",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.


In [200]:
df_financial_view.info()

<class 'pandas.DataFrame'>
RangeIndex: 30288 entries, 0 to 30287
Data columns (total 22 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   BvD ID number                      30288 non-null  str    
 1   Company name Latin alphabet        30288 non-null  str    
 2   Year                               30288 non-null  int64  
 3   avg_vol                            1064 non-null   float64
 4   std_vol                            1060 non-null   float64
 5   max_vol                            1064 non-null   float64
 6   min_vol                            1064 non-null   float64
 7   vol_stability_score                1059 non-null   float64
 8   vol_shock_ratio                    1064 non-null   float64
 9   vol_trend_slope                    1059 non-null   float64
 10  Join_Year                          30288 non-null  int64  
 11  Risk level                         30288 non-null  str    
 12  a

In [201]:
len(df_financial_view)

30288

# Stocks monthly shares

In [ ]:
df_stocks_monthly_shares = pd.read_csv("/Users/Thomas/Desktop/Master Thesis/Data/Stocks_Monthly_Shares.csv",
sep = ";")

In [213]:
df_stocks_monthly_shares.head()

,Company name Latin alphabet,Risk level_monthly_shares,BvD ID number,Monthly - Outstanding shares - January\nth Last avail. yr,Monthly - Outstanding shares - January\nth Year - 1,Monthly - Outstanding shares - January\nth Year - 2,Monthly - Outstanding shares - January\nth Year - 3,Monthly - Outstanding shares - January\nth Year - 4,Monthly - Outstanding shares - January\nth Year - 5,Monthly - Outstanding shares - January\nth Year - 6,...,Monthly - Outstanding shares - December\nth 2020,Monthly - Outstanding shares - December\nth 2019,Monthly - Outstanding shares - December\nth 2018,Monthly - Outstanding shares - December\nth 2017,Monthly - Outstanding shares - December\nth 2016,Monthly - Outstanding shares - December\nth 2015,Monthly - Outstanding shares - December\nth 2014,Monthly - Outstanding shares - December\nth 2013,Monthly - Outstanding shares - December\nth 2012,Monthly - Outstanding shares - December\nth 2011
0,Jeev Lifeworks LLP,Do not source,IN0038742908,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
1,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,MY422033-W,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
2,Daicel Corporation,Go ahead,JP4120001125937,"266942,682","276942,682","286942,682","302942,682","302942,682","302942,682","331942,682",...,"302942,682","331942,682","349942,682","349942,682","349942,682","364942,682","364942,682","364942,682","364942,682","364942,682"
3,"Polyplastics Co.,Ltd.",Take caution,JP1010401053834,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
4,"Kolon Industries, Inc.",Go ahead,KR1353110013606,"27519,091","27519,091","27519,091","27519,091","27519,091","26978,84","26978,84",...,"26978,84","26978,84","26978,84","25499,866","25151,402","25119,217","25103,951","25087,562","25055,838","25033,235"


In [215]:
shares_pattern = re.compile(r'Monthly - Outstanding shares - (\w+)\n.*(20\d{2})')
shares_cols = [c for c in df_stocks_monthly_shares.columns if shares_pattern.search(c)]

In [218]:
df_monthly_shares_long = pd.melt(
    df_stocks_monthly_shares, 
    id_vars=['BvD ID number', 'Company name Latin alphabet','Risk level_monthly_shares'], 
    value_vars=shares_cols, 
    var_name='Raw_Header', 
    value_name='Value'
)

In [219]:
df_monthly_shares_long.head()

,BvD ID number,Company name Latin alphabet,Risk level_monthly_shares,Raw_Header,Value
0,IN0038742908,Jeev Lifeworks LLP,Do not source,Monthly - Outstanding shares - January\nth 2026,n.a.
1,MY422033-W,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,Monthly - Outstanding shares - January\nth 2026,n.a.
2,JP4120001125937,Daicel Corporation,Go ahead,Monthly - Outstanding shares - January\nth 2026,"266942,682"
3,JP1010401053834,"Polyplastics Co.,Ltd.",Take caution,Monthly - Outstanding shares - January\nth 2026,n.a.
4,KR1353110013606,"Kolon Industries, Inc.",Go ahead,Monthly - Outstanding shares - January\nth 2026,"27519,091"


In [220]:
df_monthly_shares_long[['Month', 'Year']] = df_monthly_shares_long['Raw_Header'].str.extract(r'Monthly - Outstanding shares - (\w+)\n.*(20\d{2})')

In [221]:
def clean_bvd_numeric(x):
    if pd.isna(x) or str(x).lower() == 'n.a.':
        return np.nan
    # Remove dots (thousands) and replace comma with dot (decimals) if present
    # Adjust this if your specific export uses different logic
    s = str(x).replace('.', '').replace(',', '.')
    return pd.to_numeric(s, errors='coerce')

df_monthly_shares_long['Value'] = df_monthly_shares_long['Value'].apply(clean_bvd_numeric)

### Aggregate to Yearly Signals

In [225]:
df_shares_signals = df_monthly_shares_long.groupby(['BvD ID number', 'Company name Latin alphabet','Risk level_monthly_shares', 'Year'])['Value'].agg([
    ('avg_shares_outstanding', 'mean')
]).reset_index()

In [234]:
df_shares_signals.head(10)

,BvD ID number,Company name Latin alphabet,Risk level_monthly_shares,Year,avg_shares_outstanding
0,AE0001327927,Rever Events L.L.C,Take caution,2011,NaN
1,AE0001327927,Rever Events L.L.C,Take caution,2012,NaN
2,AE0001327927,Rever Events L.L.C,Take caution,2013,NaN
3,AE0001327927,Rever Events L.L.C,Take caution,2014,NaN
4,AE0001327927,Rever Events L.L.C,Take caution,2015,NaN
5,AE0001327927,Rever Events L.L.C,Take caution,2016,NaN
6,AE0001327927,Rever Events L.L.C,Take caution,2017,NaN
7,AE0001327927,Rever Events L.L.C,Take caution,2018,NaN
8,AE0001327927,Rever Events L.L.C,Take caution,2019,NaN
9,AE0001327927,Rever Events L.L.C,Take caution,2020,NaN


In [235]:
df_shares_signals['Year'] = df_shares_signals['Year'].astype(int)
df_shares_signals['Join_Year'] = df_shares_signals['Year'] + 1
df_shares_signals['BvD ID number'] = df_shares_signals['BvD ID number'].astype(str)

### Joining on financial view

In [240]:
df_stock_information = df_financial_view.merge(
    df_shares_signals[[
        'BvD ID number', 
        'Year', 
        'Join_Year', 
        'avg_shares_outstanding', 
        'Risk level_monthly_shares'
    ]],
    on=['BvD ID number', 'Year', 'Join_Year'],
    how='left'
)

In [241]:
df_stock_information.head()

,BvD ID number,Company name Latin alphabet,Year,avg_vol,std_vol,max_vol,min_vol,vol_stability_score,vol_shock_ratio,vol_trend_slope,...,supplier_sector,moodys_risk_rating,Price_trends_52 weeks_%,market_beta_1y,Earnings_per_share_DKK,Book_value_per_share_DKK,Shares outstanding,Current_market_capitalisation_DKK,avg_shares_outstanding,Risk level_monthly_shares
0,AE0001327927,Rever Events L.L.C,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"Travel, Personal & Leisure",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.,NaN,Take caution
1,AE0001327927,Rever Events L.L.C,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"Travel, Personal & Leisure",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.,NaN,Take caution
2,AE0001327927,Rever Events L.L.C,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"Travel, Personal & Leisure",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.,NaN,Take caution
3,AE0001327927,Rever Events L.L.C,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"Travel, Personal & Leisure",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.,NaN,Take caution
4,AE0001327927,Rever Events L.L.C,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"Travel, Personal & Leisure",Take caution,NaN,NaN,NaN,NaN,n.a.,n.a.,NaN,Take caution


In [243]:
df_stock_information.info()

<class 'pandas.DataFrame'>
RangeIndex: 30288 entries, 0 to 30287
Data columns (total 24 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   BvD ID number                      30288 non-null  str    
 1   Company name Latin alphabet        30288 non-null  str    
 2   Year                               30288 non-null  int64  
 3   avg_vol                            1064 non-null   float64
 4   std_vol                            1060 non-null   float64
 5   max_vol                            1064 non-null   float64
 6   min_vol                            1064 non-null   float64
 7   vol_stability_score                1059 non-null   float64
 8   vol_shock_ratio                    1064 non-null   float64
 9   vol_trend_slope                    1059 non-null   float64
 10  Join_Year                          30288 non-null  int64  
 11  Risk level                         30288 non-null  str    
 12  a

# Stocks Closing prices

In [212]:
df_stocks_closing_price = pd.read_csv("/Users/Thomas/Desktop/Master Thesis/Data/Stocks_Monthly_ClosingPrices.csv",
sep = ";")

In [245]:
df_stocks_closing_price.head()

,Company name Latin alphabet,Risk level_stock_closing_price,BvD ID number,Monthly - Closing price - January\nDKK Last avail. yr,Monthly - Closing price - January\nDKK Year - 1,Monthly - Closing price - January\nDKK Year - 2,Monthly - Closing price - January\nDKK Year - 3,Monthly - Closing price - January\nDKK Year - 4,Monthly - Closing price - January\nDKK Year - 5,Monthly - Closing price - January\nDKK Year - 6,...,Monthly - Closing price - December\nDKK 2020,Monthly - Closing price - December\nDKK 2019,Monthly - Closing price - December\nDKK 2018,Monthly - Closing price - December\nDKK 2017,Monthly - Closing price - December\nDKK 2016,Monthly - Closing price - December\nDKK 2015,Monthly - Closing price - December\nDKK 2014,Monthly - Closing price - December\nDKK 2013,Monthly - Closing price - December\nDKK 2012,Monthly - Closing price - December\nDKK 2011
0,Jeev Lifeworks LLP,Do not source,IN0038742908,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
1,Polyplastics Asia Pacific Sdn. Bhd.,Go ahead,MY422033-W,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
2,Daicel Corporation,Go ahead,JP4120001125937,"59,2730255","56,00781229","66,25364433","45,43677881","44,2562568","45,21201329","61,31837023",...,"44,01595118","64,17722542","66,52929381","70,48955907","77,89479158","102,8184322","71,95080554","44,00067517","37,07348081","34,67172288"
3,"Polyplastics Co.,Ltd.",Take caution,JP1010401053834,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,...,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
4,"Kolon Industries, Inc.",Go ahead,KR1353110013606,"238,6321357","124,1851322","198,2519124","238,2667075","336,6989996","226,1529321","235,2330992",...,"228,5519224","286,8595872","334,1039399","516,1002392","432,1496996","365,820044","269,2346805","280,5331504","336,7127564","315,7635493"


In [246]:
price_pattern = re.compile(r'Monthly - Closing price - (\w+)\n.*(20\d{2})')
price_cols = [c for c in df_stocks_closing_price.columns if price_pattern.search(c)]

In [247]:
df_price_long = pd.melt(
    df_stocks_closing_price, 
    id_vars=['BvD ID number', 'Company name Latin alphabet', 'Risk level_stock_closing_price'], 
    value_vars=price_cols, 
    var_name='Raw_Header', 
    value_name='Value'
)

In [248]:
df_price_long[['Month', 'Year']] = df_price_long['Raw_Header'].str.extract(r'Monthly - Closing price - (\w+)\n.*(20\d{2})')

In [249]:
df_price_long['Value'] = df_price_long['Value'].astype(str).str.replace(',', '.')
df_price_long['Value'] = pd.to_numeric(df_price_long['Value'], errors='coerce')

### Trend function

In [250]:
def get_rel_trend(series):
    valid = series.dropna()
    if len(valid) < 2: return np.nan
    y = valid.values
    x = np.arange(len(y))
    slope = np.polyfit(x, y, 1)[0]
    return slope / valid.mean() if valid.mean() != 0 else 0

In [251]:
df_price_signals = df_price_long.groupby([
    'BvD ID number', 
    'Company name Latin alphabet', 
    'Risk level_stock_closing_price', 
    'Year'
])['Value'].agg([
    ('avg_closing_price', 'mean'),
    ('price_volatility_score', lambda x: x.std() / x.mean() if x.mean() != 0 else 0),
    ('price_trend_slope', get_rel_trend)
]).reset_index()

In [252]:
df_price_signals['Year'] = df_price_signals['Year'].astype(int)
df_price_signals['Join_Year'] = df_price_signals['Year'] + 1
df_price_signals['BvD ID number'] = df_price_signals['BvD ID number'].astype(str)

In [256]:
df_price_signals.tail()

,BvD ID number,Company name Latin alphabet,Risk level_stock_closing_price,Year,avg_closing_price,price_volatility_score,price_trend_slope,Join_Year
30283,ZA202147957107,Aspen SA Operations (PTY) LTD,Take caution,2022,NaN,NaN,NaN,2023
30284,ZA202147957107,Aspen SA Operations (PTY) LTD,Take caution,2023,NaN,NaN,NaN,2024
30285,ZA202147957107,Aspen SA Operations (PTY) LTD,Take caution,2024,NaN,NaN,NaN,2025
30286,ZA202147957107,Aspen SA Operations (PTY) LTD,Take caution,2025,NaN,NaN,NaN,2026
30287,ZA202147957107,Aspen SA Operations (PTY) LTD,Take caution,2026,NaN,NaN,NaN,2027


# Final Join, so all dataframes are combined

In [257]:
df_stock_view = df_financial_view.merge(
    df_price_signals[[
        'BvD ID number', 
        'Year', 
        'Join_Year', 
        'avg_closing_price', 
        'price_volatility_score', 
        'price_trend_slope',
        'Risk level_stock_closing_price'
    ]],
    on=['BvD ID number', 'Year', 'Join_Year'],
    how='left'
)

In [262]:
df_stock_view.to_csv("/Users/Thomas/Desktop/Master Thesis/Data/stock_view.csv")

# contract

In [174]:
df_contract = pd.read_csv("/Users/Thomas/Desktop/Master Thesis/Data/contract_year.csv",
    engine="python",
    on_bad_lines="skip"
)

In [180]:
df_contract.head(10)

,contract_id,contract_number,contract_name,contract_status,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,...,Preferred_supplier_tag,start_year,expiry_year,open_ended_contract,end_year,start_year_capped,year,years_to_expiry,contract_age_years,expiry_pressure_bucket
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2018,9,0,5y_plus
1,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2019,8,1,5y_plus
2,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2020,7,2,5y_plus
3,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2021,6,3,5y_plus
4,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2022,5,4,3_to_5y
5,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2023,4,5,3_to_5y
6,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2024,3,6,1_to_3y
7,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2025,2,7,1_to_3y
8,8157,8158,Kalundborg Bioenergi_Master_2017_SA,published,False,fixed,2018-04-01 00:00:00+00:00,2033-07-01 00:00:00+00:00,17030,49827,...,0.0,2018,2033,False,2025,2018,2018,15,0,5y_plus
9,8157,8158,Kalundborg Bioenergi_Master_2017_SA,published,False,fixed,2018-04-01 00:00:00+00:00,2033-07-01 00:00:00+00:00,17030,49827,...,0.0,2018,2033,False,2025,2018,2019,14,1,5y_plus
